# 03 · Matriz ABC × XYZ sobre train

Solo carga días ≤ fin de train y no lee `reports/eda_five_years/`. La matriz se congela una vez (Principio V).

In [ ]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)
cfg = load_validated_config()
print("versión de la config:", cfg["version"])

In [ ]:
from src.segmentation import run_train_matrix

res = run_train_matrix(cfg)
matrix = res["matrix"]
matrix.head(20)

In [ ]:
pd.crosstab(matrix["abc_class"], matrix["xyz_class"].fillna("-"), margins=True)

Cobertura (SC-001): cada artículo activo tiene clase ABC y XYZ, o motivo de exclusión.

In [ ]:
active = matrix[matrix["exclusion_reason"] != "inactive_in_set"]
ok = active["abc_class"].notna() & (active["xyz_class"].notna() | active["exclusion_reason"].notna())
print(f"cobertura: {ok.mean():.1%} de {len(active)} artículos activos")
assert ok.all()
matrix["exclusion_reason"].value_counts(dropna=False)

Sensibilidad de los valores por defecto (±5 pp, FR-013), solo sobre train.

In [ ]:
res["sensitivity"].pivot_table(index=["dimension", "variant"], columns="class", values="count")

Verificación manual de 10 artículos (SC-003): el gasto se recalcula desde unidades × precio por tienda-día, y el CV desde bloques de 28 días, sin usar `build_matrix`.

In [ ]:
from src.config import train_end, train_years
from src.data import load_interim

end = train_end(cfg)
start = min(a for a, _ in train_years(cfg).values())
raw = load_interim(cfg, end, series_level=True)
si = raw["series_index"]
ranked = active.reset_index(drop=True)
sample = pd.concat([ranked.head(5), ranked.sample(min(5, len(ranked)), random_state=0)]).drop_duplicates("item_id")
total = ranked["spend"].sum()
checks = []
for _, row in sample.iterrows():
    rows = si.index[si["item_id"] == row["item_id"]]
    u = np.asarray(raw["units"][rows, start - 1:end], dtype=float)
    p = np.asarray(raw["prices_daily"][rows, start - 1:end], dtype=float)
    spend = np.nansum(u * p)
    cum_before = ranked.loc[ranked["spend"] > row["spend"], "spend"].sum() + ranked.loc[(ranked["spend"] == row["spend"]) & (ranked["item_id"] < row["item_id"]), "spend"].sum()
    daily = np.where(np.isnan(u).all(axis=0), np.nan, np.nansum(u, axis=0))
    blocks = daily[: len(daily) // 28 * 28].reshape(-1, 28)
    sums = blocks[~np.isnan(blocks).any(axis=1)].sum(axis=1)
    cv = sums.std(ddof=0) / sums.mean() * 100 if len(sums) >= cfg["segmentation"]["cv_min_blocks"] and sums.mean() > 0 else np.nan
    checks.append({"item_id": row["item_id"], "spend_ok": np.isclose(spend, row["spend"]),
                   "cum_pct_ok": np.isclose(cum_before / total, row["cum_pct_before"]),
                   "cv_ok": (np.isnan(cv) and np.isnan(row["cv"])) or np.isclose(cv, row["cv"]),
                   "abc_class": row["abc_class"], "xyz_class": row["xyz_class"]})
checks = pd.DataFrame(checks)
assert checks[["spend_ok", "cum_pct_ok", "cv_ok"]].all().all()
checks